In [0]:
%sql
CREATE CATALOG IF NOT EXISTS credit_transactions
MANAGED LOCATION 'abfss://data@crdtrans.dfs.core.windows.net/catalog/';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS credit_transactions.bronze;
CREATE SCHEMA IF NOT EXISTS credit_transactions.silver;
CREATE SCHEMA IF NOT EXISTS credit_transactions.gold;

In [0]:
%sql
-- Create volume for uploading 2 json files merchant and customer

CREATE VOLUME IF NOT EXISTS credit_transactions.bronze.reference_data

In [0]:
merchants=spark.read.json("/Volumes/credit_transactions/bronze/reference_data/merchants.json")
merchants.write.mode("overwrite").saveAsTable("credit_transactions.bronze.merchants")

In [0]:
customer=spark.read.json("dbfs:/Volumes/credit_transactions/bronze/reference_data/customers.json")
customer.write.mode("overwrite").saveAsTable("credit_transactions.bronze.customer")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS credit_transactions.gold.daily_fraud_metrics (
    metric_date DATE NOT NULL,
    total_transactions BIGINT,
    fraud_count BIGINT,
    fraud_rate DOUBLE,
    fraud_amount_total DOUBLE,
    fraud_count_online BIGINT,
    fraud_count_pos BIGINT,
    fraud_count_atm BIGINT
)
USING DELTA

In [0]:
%sql
ALTER TABLE credit_transactions.gold.daily_fraud_metrics
ADD CONSTRAINT pk_daily_fraud_metrics PRIMARY KEY (metric_date)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS credit_transactions.gold.daily_business_metrics (
    metric_date DATE NOT NULL,
    total_transactions BIGINT,
    total_volume DOUBLE,
    avg_ticket_size DOUBLE,
    active_customers BIGINT,
    active_merchants BIGINT,
    volume_credit_card DOUBLE,
    volume_debit_card DOUBLE,
    volume_digital_wallet DOUBLE
)
USING DELTA

In [0]:
%sql
ALTER TABLE credit_transactions.gold.daily_business_metrics
ADD CONSTRAINT pk_daily_business_metrics PRIMARY KEY (metric_date)